# How to calibrate the geometry

Calibration has two steps, as with ImageD11's notebooks:

1. **Powder rings** (`python -m anri.calib rings`): from the motors' first guess, fit the distance, the beam centre and the two tilts that move rings (tilt_y, tilt_z) so that a known phase's rings fall at their 2θ, as ImageD11's `powder_calib` notebook does. The data are the scan's own sparse pixels summed over every frame (or a dense powder image). A search finds the centre and the distance first, without assigning rings, so the start can be far off: the centre 100 pixels away, the distance 10% out. The wavelength stays as given: rings alone cannot separate it from the distance, which absorbs its error. The result is a geometry good enough to index.
2. **The scan** (`python -m anri.calib fit`): with the sample indexed (the fit stage of `python -m anri.index`), fit the distance, beam centre, three tilts and the wavelength (or the lattice scale) to the peaks. Each peak is matched to an orientation of the map and a reflection, and scatters from where the beam crosses that orientation's voxels at the peak's dty and ω: the parallax a single-grain fit (ImageD11's `makemap`, `refine_em`, its silicon notebook) cannot model on a scan. The peaks' ω separate the wavelength from the distance. Each orientation's rotation is refitted with the geometry.

This notebook runs both stages, and the indexing between them, with the plots that help you choose their options and judge the results.

| Section | Options | What the plots show |
|---|---|---|
| 0. Powder rings | `ring_tols`, `max_shift`, `span` (and the data: `image`, `scans`, `threshold`, `detector_mask`, `spatial`) | the centre and distance searches, and each ring's offset per η sector before and after the fit |
| Indexing | | the indexer's stages, with the rings' geometry |
| 1. The map | `--min-occ` | how much of the map's occupancy each cut leaves out, voxel by voxel |
| 2. Matching | `--tols`, `--lattice` | how far each peak's (h, k, l) is from integers at the starting geometry: real matches against chance |
| 3. The fit | `--free`, `--iter`, `--clip`, `--extra-rounds`, `--row-step` | each round's matched peaks, residuals and parameters, and how far it moved them (the stage's log) |
| 4. Residuals | `--clip` | the residuals against ω, the azimuth on the detector and dty, and their distribution |
| 5. Strain left | `--min-peaks` | each orientation's strain with its lattice free and the geometry fixed: the calibration's precision |
| 6. Files | | the ImageD11 parameter files (`python -m anri.calib pars`) and the commands that remake the calibration from the dataset |

Where a choice is cheap to redo, a slider redraws the plot, and **the value on the slider is what the later cells use**. The stages are the command line's own functions (`anri.calib.stages.stage_rings`, `stage_fit`, `anri.index.stages`), so the files written here are those the command line writes, with the same provenance (`python -m anri.calib show <file>`). The last cell prints the commands that remake the calibration from the dataset: rings, pars, the indexing and the fit.

What you need: an ImageD11 DataSet of the scan (its sparse pixels and its peaks table, the DataSet's `pksfile`) and a `pars.json` with the phase and the first guess of the geometry (`dataset_file`, `parfile`); or, to skip the rings and the indexing, an indexed map (`fit_file`: the fit stage, not the local search) made with a geometry good enough to index. Wedge, chi and y0 stay as they were indexed.

**The sample must be a calibrant or little deformed.** Each peak is matched to the orientation that predicts it best at the current geometry. Where a voxel holds orientations closer to one another than the geometry's error (cells, subgrains, a local search's refinements), that choice absorbs part of the error, and each round closes only part of the gap: on the cells of the 316L phantom (a few tenths of a degree) the distance was still 6e-4 out after ten rounds. The stage warns when its last round still moved the geometry. Calibrate on a calibrant (an undeformed Si cube, CeO2) or a well-annealed sample; a deformed sample's geometry is refined with its strains instead. A calibrant (known lattice, e.g. a silicon or CeO2 crystal) frees the wavelength; any other sample (known wavelength) frees the lattice scale instead: the two are degenerate.

Run it top to bottom; after moving a slider, re-run the cells below it. Figures are interactive (ipympl, `%matplotlib widget`). It needs `ipywidgets`, `ipympl` and `matplotlib`. For the figures to survive a restart of the Jupyter server, turn on JupyterLab's *Save Widget State Automatically* ([how](index.rst#figures-that-survive-a-restart)). `interactive = False` draws each figure once, at the parameters' values, as an inline image instead, as for these docs; the sliders then stay at their starting values, which the later cells use.

With neither given, it renders the grains of the 316L [phantom](../tutorials/phantom.ipynb) of the [indexing tutorial](../tutorials/indexing.ipynb), undeformed, and writes its parameters with a first guess far from its true geometry. It then runs the whole chain (several minutes on a laptop CPU; it needs ImageD11), so it runs out of the box and the result can be checked against the true geometry.


In [ ]:
# Parameters: the options of python -m anri.calib rings and fit, at their defaults. The sliders below start
# from these values.

# what to start from; all None renders a phantom with a first guess far from its true geometry
dataset_file = None  # an ImageD11 DataSet (..._dataset.h5): rings, indexing and calibration
parfile = None  # pars.json: the phase and the first guess of the geometry (None: the DataSet's)
fit_file = None  # an anri.index fit stage file instead: skip the rings and the indexing
peaks = None  # --peaks: ImageD11 peaks table (None: the DataSet's pksfile)
lattice = None  # --lattice: reference (a, b, c, alpha, beta, gamma), e.g. a calibrant's certified cell (None: the phase's)
outdir = None  # where the stage files go (None: a new temporary folder)

# 0. powder rings
ring_free = ("distance", "y_center", "z_center", "tilt_y", "tilt_z")  # --free of anri.calib rings
ring_tols = (0.2, 0.1)  # --tols: 2theta window of each round (deg), a few pixels wide at least
search = True  # search for the centre and the distance first (--no-search: the start is close)
max_shift = 200.0  # --max-shift: centre search range (pixels)
span = (0.7, 1.4)  # --span: distance search range (x the start)
image = None  # a dense powder image (e.g. a calibrant's, background subtracted) instead of the scan's own pixels
ring_phase = None  # --phase: the phase whose rings are fitted (None: the only one in parfile)
detector_mask = None  # --mask: non-zero where masked (None: the DataSet's maskfile)
spatial = None  # --spatial: (e2dx, e2dy) distortion files, for an image (the DataSet's own are used for its pixels)
scans = None  # --scans: sum only this many of the DataSet's scans, spread evenly (None: all)
threshold = 0.0  # --threshold: keep pixels above this, with it subtracted
ring_iter = 30  # --iter: Gauss-Newton iterations per round at most
n_eta = 360  # --n-eta: eta sectors, one residual per (ring, sector)
ring_clip = 4.0  # --clip: leave out sectors beyond this many sigma
merge = 1e-4  # --merge: reflections within this d* (1/angstrom) are one ring

# 1. the map
min_occ = 0.01  # --min-occ: leave out occupancies below this x the largest
# 2. matching
tols = (0.1, 0.05, 0.025, 0.025)  # --tols: (h, k, l) matching tolerance of each round
# 3. the fit
free = ("distance", "y_center", "z_center", "tilt_x", "tilt_y", "tilt_z", "wavelength")  # --free: "lattice"
# instead of "wavelength" for a sample that is not a calibrant
n_iter = 10  # --iter: Gauss-Newton iterations per round
clip = 4.0  # --clip: leave out peaks beyond this many (robust) sigma
extra_rounds = 6  # --extra-rounds: repeat the last tolerance while a round moves the geometry by more than its errors
row_step = 1  # --row-step: use every Nth dty row's peaks (e.g. 10 on a CPU)
# 5. strain left
min_peaks = 20  # --min-peaks: free the lattice of orientations with at least this many peaks

n_cpu = 4  # --n-cpu
out = None  # --out (None: the map's name with _calib, beside it)
pars_folder = None  # where python -m anri.calib pars writes (None: calib_pars beside the calibration)

# this notebook only
interactive = True  # sliders (ipympl); False: static figures at the parameters' values, which survive a
# restart of the Jupyter server and show in the docs


In [ ]:
import anri.utils

anri.utils.setup(n_cpu=n_cpu)  # before any JAX computation

import os
import shlex
import tempfile
import time
from types import SimpleNamespace

import h5py
import ipywidgets as widgets
import numpy as np
from IPython import get_ipython
from IPython.display import display
from matplotlib import pyplot as plt

import anri.calib as cb
import anri.index as ix
import anri.io
import anri.phantom

get_ipython().run_line_magic("matplotlib", "widget" if interactive else "inline")
T0 = time.time()
NOTEBOOK = "calibration.ipynb"  # recorded as the command in each stage file's provenance
true_pars = None  # the phantom's true geometry, to check the result against


def log(msg):
    print(f"[{time.time() - T0:7.1f} s] {msg}", flush=True)


def figure(name, **kwargs):
    # a figure shown where display() puts it (under its sliders), not at the end of the cell; re-running the cell
    # replaces it
    plt.close(name)
    with plt.ioff():
        fig = plt.figure(name, layout="constrained", **kwargs)
    if interactive:
        fig.canvas.header_visible = False
    return fig


def show(fig):
    display(fig.canvas if interactive else fig)


def interact(fig, update, **controls):
    # sliders that call a plot's update, which redraws it and returns a summary; their values are in .kwargs
    if not interactive:  # draw once at the controls' starting values
        values = {k: c.value for k, c in controls.items()}
        print(update(**values))
        display(fig)
        return SimpleNamespace(kwargs=values)
    w = widgets.interactive(lambda **kw: print(update(**kw)), **controls)
    display(w, fig.canvas)
    return w


def slider(value, lo, hi, step, name, log_scale=False):
    kind = widgets.FloatLogSlider if log_scale else widgets.FloatSlider
    extra = {"base": 10} if log_scale else {"readout_format": ".3g"}
    return kind(value=value, min=lo, max=hi, step=step, description=name, continuous_update=False, **extra)

## The phantom

With no data given, the grains of the 316L phantom of the [indexing tutorial](../tutorials/indexing.ipynb), without its cells and twins, are rendered into a temporary folder as an ImageD11 dataset, with its true geometry, and ImageD11 segments its pixels into a peaks table. Its `pars.json` gets a first guess as an experiment's would be: the distance from a motor 7% short, the wavelength from a wrong energy (1.8% short), the beam centre 60 and 90 pixels off, the tilts 1 to 2 mrad off. No ring falls within the indexer's windows with that geometry. Each grain has one orientation and there is no strain, so it behaves like a polycrystalline calibrant.


In [ ]:
if fit_file is None and dataset_file is None:
    # the 316L phantom's grains (tests/data/phantoms/am316l), without its cells and twins: an undeformed sample
    a = 3.5966
    ph = anri.phantom.polycrystal(n=103, step=0.5, radius=25.0, n_grains=40, cell_spread_deg=0.0, twin_grains=0, seed=0)
    truth = anri.phantom.tensormap(ph, [a, a, a, 90.0, 90.0, 90.0], 225, "316L", 0.5)
    wl = 0.2843
    true_pars = {
        "y_center": 1023.5, "y_size": 75.0, "tilt_y": 1e-3, "z_center": 1023.5, "z_size": 75.0, "tilt_z": -2e-3,
        "tilt_x": 0.0, "distance": 150e3, "o11": -1, "o12": 0, "o21": 0, "o22": -1, "wavelength": wl, "wedge": 0.0,
        "chi": 0.0, "omegasign": 1.0, "t_x": 0.0, "t_y": 0.0, "t_z": 0.0,
    }
    y0_true = 0.3
    geom_r = anri.io.imaged11.geom_from_pars(true_pars, y0_true, wl * 2e-4 / 2.355, 5e-5, 5e-5, sig_beam=1.4 / 2.355,
                                             voxel_size=0.5, sig_psf=0.5)
    entries = anri.io.imaged11.entries_from_tensormap(truth)
    entries["density"] = np.full(len(entries["pos"]), 30.0)  # counts scale
    rings8 = ix.data.ring_table(np.array([a, a, a, 90.0, 90.0, 90.0]), 225, wl, 8)
    omega_r, dty_r = anri.io.imaged11.motor_grid((0.0, 180.0), 0.1, (y0_true - 30.0, y0_true + 30.0), 1.0)
    root, sample, dataset = tempfile.mkdtemp(), "phantom", "am316l"
    sparse = os.path.join(root, "phantom_sparse.h5")
    t0 = time.time()
    anri.io.imaged11.simulate_sparse(sparse, entries, rings8["hkls"], np.ones(len(rings8["hkls"])), geom_r, omega_r,
                                     dty_r, (2048, 2048), max_frames=61)
    log(f"phantom rendered in {time.time() - t0:.0f} s into {root}")

    # the experiment's first guess: the distance 7% short, the wavelength 1.8% short, the centre 60 and 90 px off
    start = dict(true_pars, distance=140e3, wavelength=wl * 0.982, y_center=1083.5, z_center=933.5, tilt_x=1e-3,
                 tilt_y=-1e-3, tilt_z=0.0)
    cell_r = {"cell__a": a, "cell__b": a, "cell__c": a, "cell_alpha": 90.0, "cell_beta": 90.0, "cell_gamma": 90.0,
              "cell_lattice_[P,A,B,C,I,F,R]": 225}
    parfile = anri.io.imaged11.write_pars(os.path.join(root, "pars"), start, {"316L": cell_r})
    dataset_file = anri.io.imaged11.write_dataset(sparse, root, sample, dataset, y0=y0_true, parfile=parfile)
    peaks = anri.io.imaged11.write_peaks_table(dataset_file, nproc=n_cpu)  # ImageD11's segmentation into 2D and 4D peaks
    log(f"peaks table: {peaks}")
    outdir = outdir or root


## 0. Powder rings: `ring_tols`, `max_shift`, `span`

`anri.calib.stages.stage_rings`, as `python -m anri.calib rings` runs it, on the DataSet's sparse pixels summed over every frame (a scan's spots fill its rings well enough; `scans` sums fewer, for speed), or on a dense powder image (`image`, e.g. a CeO2 calibrant's, as ImageD11's notebook uses; it needs `parfile`, and `spatial` for its distortion). Pixels at or below `threshold` are left out, as are those `detector_mask` masks (default: the DataSet's). Each pixel's 2θ is seen from the lab origin, as ImageD11 computes it.

First the searches, which assign no rings. The centre is where the histogram of the pixels' radii is sharpest: on the left plot a single bright spot, which the finer grids close in on. The distance is the one whose ring windows catch the most intensity above a flat background: one clear peak on the score curve. Two peaks of similar height would mean the rings could be matched one ring off: narrow `span` around the right one. A centre more than `max_shift` from the start is not searched.

Then the fit (Levenberg-Marquardt in rounds of `ring_tols`, the pixels re-assigned to the nearest ring at every iteration): one residual per (ring, 1° η sector), the sector's mean 2θ minus its ring's. On the last plot every ring should lie flat at zero after the fit. A sine in η left over, the same for every ring, is a centre error; one growing with 2θ, a tilt; a constant growing with 2θ, the distance. `ring_tols` must stay a few pixels wide (one Eiger pixel at 150 mm is 0.03°); rings closer than twice the widest to a neighbour are left out, and reflections closer in d* than `merge` are one ring. Sectors further than `ring_clip` standard deviations from their ring are left out of each round, and each round stops after at most `ring_iter` iterations.

The parameters are written as ImageD11 parameter files (`python -m anri.calib pars`), which the indexer then uses.


In [ ]:
if fit_file is None:
    outdir = outdir or tempfile.mkdtemp()
    stem = os.path.basename(dataset_file)[: -len("_dataset.h5")]
    if image is not None:  # an image needs the starting geometry and the phase: the DataSet's, unless given
        parfile = parfile or anri.io.imaged11.read_dataset(dataset_file)["parfile"]
    rings_file = cb.stages.stage_rings(image or dataset_file, os.path.join(outdir, f"{stem}_rings.h5"),
                                       parfile=parfile, phase=ring_phase, detector_mask=detector_mask,
                                       spatial=spatial, scans=scans, threshold=threshold, free=tuple(ring_free),
                                       tols=tuple(ring_tols), n_iter=ring_iter, n_eta=n_eta, clip=ring_clip,
                                       merge=merge, search=search, max_shift=max_shift, span=tuple(span),
                                       command=NOTEBOOK, log=log)
    rd = cb.diagnostics.read_rings(rings_file)
    if "centre" in rd:
        fig = figure("centre_search", figsize=(14, 4.2))
        cb.plots.plot_centre_search(fig, rd)
        show(fig)
        fig = figure("distance_search", figsize=(9, 3.5))
        cb.plots.plot_distance_search(fig, rd)
        show(fig)
    fig = figure("ring_offsets", figsize=(14, 4.5))
    cb.plots.plot_ring_offsets(fig, rd)
    show(fig)

    ring_parfile = cb.stages.write_pars(rings_file, os.path.join(outdir, "rings_pars"))
    print(f"-> {ring_parfile}\n")
    print(f"{'':12s}{'start':>14s}{'rings':>14s}{'+-':>10s}" + ("" if true_pars is None else f"{'truth':>14s}"))
    for k in (*ring_free, "wavelength"):
        line = f"{k:12s}{rd['start'][k]:14.7g}{rd['geometry'][k]:14.7g}{rd['err'].get(k, 0.0):10.2g}"
        print(line + ("" if true_pars is None else f"{true_pars[k]:14.7g}"))


## Indexing

The indexer's stages at their defaults, with the rings' geometry (`python -m anri.index histogram ... --parfile <rings_pars/pars.json>`, then search, filter and fit; the [indexing parameters](indexing_parameters.ipynb) guide shows how to choose their options). The `pars.json` written from the rings names the rings file, so the histogram stage records it as its source: the calibration's provenance then goes back through the indexing and the rings to the dataset. The calibration takes the fit stage's map, not the local search's: the local search's orientations, a fraction of a degree apart, would let each round's matching keep part of the geometry's error, and the calibration would converge very slowly.


In [ ]:
if fit_file is None:
    sample_dir = os.path.dirname(os.path.dirname(os.path.abspath(dataset_file)))  # ImageD11's layout
    sample_name = os.path.basename(sample_dir)
    histogram_file = ix.stages.stage_histogram(os.path.dirname(sample_dir), sample_name, stem[len(sample_name) + 1 :],
                                               os.path.join(outdir, f"{stem}_histogram.h5"), parfile=ring_parfile,
                                               command=NOTEBOOK, log=log)
    search_file = ix.stages.stage_search(histogram_file, command=NOTEBOOK, log=log)
    filter_file = ix.stages.stage_filter(search_file, command=NOTEBOOK, log=log)
    fit_file = ix.stages.stage_fit(filter_file, command=NOTEBOOK, log=log)


## 1. The map: `--min-occ`

Each voxel of the map holds up to `cand` candidate orientations (64 by default), most of them with almost no occupancy. A peak is matched to an orientation that a voxel along its beam path holds, so weak candidates give more chances to match a peak to the wrong orientation by chance; `min_occ` leaves out every occupancy below that fraction of the largest in the map.

The histogram of all occupancies should show the occupied orientations well apart from a floor of near-zero candidates: put the cut in the gap. On the right, the share of each voxel's occupancy that is kept: it should stay near 1 inside the grains. A voxel whose occupancy is all left out cannot place any peak, and peaks from it will not match.

The numbers come from `anri.calib.diagnostics.read_occupancy` and `occupancy_kept` (the stage's own cut, `anri.calib.stages.select_occupancy`); the plot is `anri.calib.plots.plot_min_occ`.

In [ ]:
occ = cb.diagnostics.read_occupancy(fit_file)
fig = figure("min_occ", figsize=(13, 4.2))
w_occ = interact(fig, cb.plots.plot_min_occ(fig, occ, min_occ),
                 min_occ=slider(min_occ, -5, 0, 0.05, "min_occ", log_scale=True))

## 2. Matching: `--tols`, `--lattice`

The stage reads the peaks (ImageD11's 2D peaks merged per 4D label and dty row, `anri.io.imaged11.read_peaks`) and the map (`anri.calib.stages.load_inputs`), and matches every peak at the starting geometry (`anri.calib.fit.match`): the beam's centre line at the peak's dty and ω is sampled at every voxel it crosses, each orientation held there gives the peak's (h, k, l), and the match is the orientation and point whose (h, k, l) is nearest to integers. The fit uses the peaks whose distance from integers is within the tolerance of each round, and after each round matches again with the new geometry and orientations, so the tolerances can shrink from round to round.

What to look for:

- **Real matches pile up near 0**, above a broad floor of chance matches (the nearest of many orientations and points along the ray is always somewhat near integers). The first tolerance should take in the whole pile at the starting geometry, and not much of the floor. A wrong geometry, or orientations off by up to half the indexing grid step, widen the pile: the larger (h, k, l), the wider, as the scatter against intensity shows for weak (often high-angle) peaks.
- **No pile at all**: the geometry is too far off, the map's lattice is wrong (`--lattice`), or the peaks are not this scan's.
- The later tolerances (the grey lines) apply after the geometry has been fitted once; section 4 shows the distances again with the calibrated geometry.

The slider sets the first round's tolerance; the later rounds keep theirs from the parameters.

The distances come from `anri.calib.diagnostics.match_distances`; the plot is `anri.calib.plots.plot_match`.

In [ ]:
d = cb.stages.load_inputs(fit_file, peaks, lattice, w_occ.kwargs["min_occ"])
t0 = time.time()
dist_start = cb.diagnostics.match_distances(d)  # at the starting geometry
log(f"{dist_start['n_peaks']} peaks from {d['peaks_file']} matched in {time.time() - t0:.0f} s; "
    f"{len(dist_start['err'])} cross the map")
fig = figure("match", figsize=(14, 4))
w_tol = interact(fig, cb.plots.plot_match(fig, dist_start, tols), tol=slider(tols[0], 0.005, 0.4, 0.005, "tol"))

## 3. The fit: `--free`, `--iter`, `--clip`

The stage itself, as `python -m anri.calib fit` runs it. Each round matches the peaks, keeps one peak per (orientation, reflection, branch, dty row), leaves out those further than `clip` robust standard deviations from their prediction, and fits the free parameters and every orientation's rotation by Levenberg-Marquardt (`anri.calib.fit.calibrate`). Then, with the geometry fixed, each orientation with at least `min_peaks` peaks gets its lattice freed (`anri.calib.fit.strain_left`, section 5).

In the log, from round to round: the matched peaks should grow or stay, the standard deviations (sigma) in sc, fc and ω should fall to the peaks' own precision, and the parameters settle. Each round prints how far it moved each free parameter, in its error bars; after the listed tolerances the stage repeats the last one while a round still moves any by more than 1 (at most `extra_rounds` times), and warns if it had to stop before that: its error bars are then too small. On a CPU, `row_step` (e.g. 10) fits the peaks of every 10th dty row only: matching costs in proportion to the peaks, and on a phantom every 10th row gave the geometry of all rows.

`anri.calib.diagnostics.read_calib` reads the result back for the sections below.

In [ ]:
tols_used = (w_tol.kwargs["tol"], *tols[1:])
min_occ_used = w_occ.kwargs["min_occ"]
calib_file = cb.stages.stage_fit(fit_file, out, peaks=peaks, lattice=lattice, free=tuple(free), tols=tols_used,
                                 n_iter=n_iter, clip=clip, extra_rounds=extra_rounds, row_step=row_step,
                                 min_occ=min_occ_used, min_peaks=min_peaks,
                                 command=NOTEBOOK, log=log)
c = cb.diagnostics.read_calib(calib_file)

print(f"{'':12s}{'start':>14s}{'fitted':>14s}{'+-':>10s}{'moved / +-':>12s}" + ("" if true_pars is None else f"{'truth':>14s}{'off / +-':>10s}"))
for k in c["free"]:
    v0, v, e = c["start"][k], c["fitted"][k], c["err"][k]
    line = f"{k:12s}{v0:14.7g}{v:14.7g}{e:10.2g}{(v - v0) / e:12.1f}"
    if true_pars is not None:
        t = 1.0 if k == "lattice" else true_pars[k]
        line += f"{t:14.7g}{(v - t) / e:10.1f}"
    print(line)

## 4. Residuals: `--clip`

Each matched peak's observed − predicted (sc, fc, ω), in units of the standard deviations the last round measured, against ω, the azimuth on the detector (about the fitted beam centre) and dty. The red lines are medians in bins. With the right geometry they are flat at zero, and what is left is the peaks' noise.

- **A sinusoid against azimuth** (one period around the detector): the beam centre or tilts are not right; with them free, a pattern of higher order is detector distortion (check the DataSet's distortion files).
- **A trend against ω**: something that turns with the sample and is fixed here, the wedge or chi, or a wobble of the rotation axis.
- **A trend against dty**: y0 (fixed here, from the map), or the parallax of large grains placed wrongly by the map.
- **Heavy tails** in the histograms (beyond the Gaussian): wrong matches or overlapping peaks. A lower `clip` or tighter `tols` leave out more of them; the clip lines show where the last round cut.

Below them, the distances of (h, k, l) from integers again, with the calibrated geometry and orientations: the pile of real matches should be much narrower than at the start, and the later tolerances can be set from it.

The numbers come from `anri.calib.diagnostics.residual_trends` and `match_distances`; the plots are `anri.calib.plots.plot_residuals` and `plot_residual_hists`.

In [ ]:
trends = cb.diagnostics.residual_trends(c)
sig = c["sig"]
print(f"{len(trends['r'])} peaks; sigma (sc, fc, omega): {sig[0]:.3f} px, {sig[1]:.3f} px, {sig[2]:.4f}°; "
      f"rms residual / sigma: {trends['rms'].round(2)}")
fig = figure("residuals", figsize=(15, 9))
cb.plots.plot_residuals(fig, trends, clip)
show(fig)

dist_fit = cb.diagnostics.match_distances(d, c["ubi"], c["pars"])  # at the calibrated geometry
fig = figure("residual_hist", figsize=(15, 3.4))
cb.plots.plot_residual_hists(fig, trends, clip, dist_start, dist_fit, tols_used)
show(fig)

## 5. Strain left: `--min-peaks`

With the geometry fixed, each orientation with at least `min_peaks` matched peaks gets its lattice freed (six strains with its rotation), as ImageD11's silicon notebook estimates its precision. On a strain-free calibrant, a right geometry leaves no strain beyond the noise:

- **A deviatoric strain shared by all orientations** (its peak-weighted mean, the black line, away from zero) shows the detector's tilts or centre; a **hydrostatic** one the wavelength or distance against the reference lattice. (With the lattice scale free, the hydrostatic mean is absorbed by the scale.)
- **The scatter between orientations** is the noise of each one's fit, and falls with its number of peaks. Orientations far outside it, with many peaks, are deformed grains, or orientations that took another's peaks.
- `chi2 x` in the log is how much freeing the lattices lowered the χ²: close to 1, the geometry explains the peaks as well as free lattices do.

On a sample that is not strain-free the strain left is the sample's, plus the calibration's error.

The numbers come from `anri.calib.diagnostics.strain_left` (as `anri.calib.fit.split_strain`, which the stage logs); the plot is `anri.calib.plots.plot_strain_left`.

In [ ]:
st = cb.diagnostics.strain_left(c)
fig = figure("strain", figsize=(15, 4))
cb.plots.plot_strain_left(fig, st)
show(fig)
print(f"{st['fitted'].sum()} of {len(st['fitted'])} orientations with at least {min_peaks} peaks")
for k, v in c["results"].items():
    print(f"  {k}: {v}")

## 6. The parameter files, and the command line

`anri.calib.stages.write_pars` writes the calibration as ImageD11 parameter files (`geometry.par`, `<phase>.par` and `pars.json`; the phase's cell scaled by the fitted lattice scale if it was free), as `python -m anri.calib pars` does. Index again with them (`python -m anri.index ... --parfile <pars.json>`, or `parfile` in the [indexing parameters](indexing_parameters.ipynb) guide), or give them to ImageD11.

Then the commands that remake the calibration from the dataset, with every option each stage recorded (`anri.calib.stages.commands`): `python -m anri.calib rings`, `pars`, the indexer's `run` and `python -m anri.calib fit`, and the `pars` that writes the calibrated parameters. Last, how the file was made, all the way back to the dataset (`anri.io.provenance.show`, as `python -m anri.calib show`). With `fit_file` given, the chain starts at that indexing.

In [ ]:
pars_folder = pars_folder or os.path.join(os.path.dirname(os.path.abspath(calib_file)), "calib_pars")
new_parfile = cb.stages.write_pars(calib_file, pars_folder)
print(f"-> {new_parfile}", end="\n\n")
print("\n".join(cb.stages.commands(calib_file)))
print(shlex.join(["python", "-m", "anri.calib", "pars", calib_file, pars_folder]), end="\n\n")
print(anri.io.provenance.show(calib_file))